This file writes to the Run Tracker excel: 
- idx_max = number of samples in the resampled dataset (sampling rate = 3) from idx_settled to the end (max evalable samples)
- max_time = time_resampled[-1]/60, i.e. the minutes of experiment available

NB: BEFORE RUNNING IT CHECK THAT max_idx AND max_time ARE COLUMNS 28 AND 29

Reset variables

In [1]:
%reset -f

Import packages

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
from scipy.signal import lfilter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from openpyxl import load_workbook

# CLASSIFIERS
#from aeon.classification.distance_based import KNeighborsTimeSeriesClassifier
#from aeon.classification.interval_based import TimeSeriesForestClassifier
# from aeon.classification.sklearn import RotationForestClassifier
from sklearn.ensemble import RandomForestClassifier
#from aeon.transformations.collection.shapelet_transform import RandomShapeletTransform

from titan import functions
from titan.load_and_preprocessing import titan_load_and_preprocessing
from titan.plt_Experiment_summary import titan_plt_summary
import titan.preprocessing_resample as resample
from titan.processing_DNA import titan_plt_infl


Set user

In [3]:
user = "costanza"
# user = "calista"

Import
- Matthew Data
- Calista Data
- negative data

Create
- ``all_labels`` is a np.array 
- ``all_wells1d`` is a np.array with shape (wells x T) where T is the number of samples
- ``all_wells2d`` is a list of np arrays [P1xT, P2xT, P3xT,...] where P1,P2,P3,.. is the number of active pixels in each well, T is the number of samples


WARNING: THIS NOW ONLY GOES THROUGH 5 EXPERIMENTS FOR DEBUGGING

In [ ]:
if user == "costanza":
    onedrive_path = Path("..", "..", "..", "..", "Costanza", "OneDrive - Imperial College London")
elif user == "calista":
    onedrive_path = Path("..", "..", "..", "..", "..", "Costanza", "OneDrive - Imperial College London")  # calista you can put your directory here

excel_path = Path(onedrive_path, "Master Data Folder", "Run Tracker.xlsx")
excel_df = pd.read_excel(excel_path, sheet_name="Master Run")

# Load the workbook and select the sheet
wb = load_workbook(Path(onedrive_path, "Master Data Folder", "Run Tracker.xlsx"))
ws = wb['Master Run']

exp_folders = [Path(onedrive_path, "Master Data Folder", "Matthew Run Data"),
               Path(onedrive_path, "Master Data Folder", "Calista Run Data"),
               Path(onedrive_path, "Master Data Folder", "negative test")]

# exp_folders = [Path(onedrive_path, "Master Data Folder", "Matthew Run Data")]

all_wells2d = []

for i_exp_folder, exp_folder in enumerate(exp_folders):
    exp_paths = [f for f in exp_folder.glob('*') if f.is_dir()]
    print(f"\n\n\nDEBUG: EXP_PATHS in exp_folder: {exp_folder}")
    for i_path in range(len(exp_paths)):
        print(f"i_path {i_path}, path {exp_paths[i_path]}")

    for i_path, exp_path in enumerate(exp_paths):
        path_readout_str = str(exp_path)
        exp_id = path_readout_str[path_readout_str.rfind('D'):]
        print(f"\n-------\nDEBUG:FOLDER N {i_exp_folder}; EXP N {i_path}; EXP_ID {exp_id}")

        exp_row = excel_df[excel_df["File Name"] == exp_id]
        if exp_row["No. Of Wells"].size == 0:
            raise "Experiment not in excel"
        if exp_row["No. Of Wells"].size > 1:
            raise "More than one line in Excel corresponding to this experiment"
        n_wells = int(exp_row["No. Of Wells"])
        n_a_type = np.array(exp_row["Version No."])[0]

        exp = titan_load_and_preprocessing(exp_path, n_wells=n_wells, start_type="temperature",
                                           end_time_min=40, n_a_type=n_a_type,
                                           print_status=False, plt_gain_calib=False, save_gain_calib=False)

        for i_well, well in enumerate(exp.wells_list):
            
            well_label = exp_row["W"+str(i_well)+"T"].item()
            
            if well_label == "P" or well_label == "N":
                time_resampled, x1d_resampled = resample.resample1d(well.time, well.well_2d_bs_active_mean_filt())
                _, x2d_resampled = resample.resample2d(well.time, well.well_2d_bs_active_filt())  
                # print(f"------------------------------->>>> RESAMPLED NTIMES {time_resampled.shape} T_SETTLED={time_resampled[0]} --- MAX TIME {time_resampled[-1]/60}min")

                for row in ws.iter_rows(min_row=2):  # Skip header
                    if row[0].value == exp_id:        
                        row[28].value = time_resampled.shape[0]
                        row[29].value = round(time_resampled[-1]/60, 2)
                        row[30].value = round(time_resampled[0], 2)
                                
                if 'all_wells1d' not in locals():
                    all_wells1d = x1d_resampled[:550].reshape(1, -1)
                    all_labels = np.array([well_label])
                    all_wells2d.append(x2d_resampled[:550, :].T)
                    # print(f"DEBUG data structures CREATED - all_wells1d.shape {all_wells1d.shape}, all_labels.shape {all_labels.shape}")
                else:
                    all_wells1d = np.append(all_wells1d, x1d_resampled[:550].reshape(1, -1), axis=0)  # TODO: modify IDX END
                    all_labels = np.append(all_labels, well_label)
                    all_wells2d.append(x2d_resampled[:550, :].T)
                    # print(f"DEBUG data structures extended - all_wells1d.shape {all_wells1d.shape}, all_labels.shape {all_labels.shape}, len(all_wells2d) {len(all_wells2d)}, all_wells2d[-1].shape {all_wells2d[-1].shape}")

# Save the changes
wb.save(Path(onedrive_path, "Master Data Folder", "Run Tracker.xlsx"))




DEBUG: EXP_PATHS in exp_folder: ..\..\..\..\Costanza\OneDrive - Imperial College London\Master Data Folder\Matthew Run Data
i_path 0, path ..\..\..\..\Costanza\OneDrive - Imperial College London\Master Data Folder\Matthew Run Data\D20240328_E00_C00_F4500KHz_U_Lambda_T3
i_path 1, path ..\..\..\..\Costanza\OneDrive - Imperial College London\Master Data Folder\Matthew Run Data\D20240404_E00_C00_F4500KHz_U_Lambda_T5
i_path 2, path ..\..\..\..\Costanza\OneDrive - Imperial College London\Master Data Folder\Matthew Run Data\D20240404_E00_C00_F4500KHz_U_Lambda_T6
i_path 3, path ..\..\..\..\Costanza\OneDrive - Imperial College London\Master Data Folder\Matthew Run Data\D20240405_E00_C00_F4500KHz_U_Lambda_T9
i_path 4, path ..\..\..\..\Costanza\OneDrive - Imperial College London\Master Data Folder\Matthew Run Data\D20240409_E00_C00_F4500KHz_U_Lambda_Neg_2
i_path 5, path ..\..\..\..\Costanza\OneDrive - Imperial College London\Master Data Folder\Matthew Run Data\D20240430_E00_C00_F4500KHz_U_Lamb

C:\Users\Costanza\AppData\Local\Temp\ipykernel_21036\2430702868.py:37: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  n_wells = int(exp_row["No. Of Wells"])
C:\Users\Costanza\Documents\PhD\titan-processing-costanza\titan\preprocessing_functions.py:115: RuntimeWarning: divide by zero encountered in log
  log_div = np.log((tau_i - C) / A) / np.log((tau_f - C) / A)  # Intermediate operation
C:\Users\Costanza\Documents\PhD\titan-processing-costanza\titan\preprocessing_functions.py:115: RuntimeWarning: invalid value encountered in divide
  log_div = np.log((tau_i - C) / A) / np.log((tau_f - C) / A)  # Intermediate operation
C:\Users\Costanza\Documents\PhD\titan-processing-costanza\titan\preprocessing_functions.py:116: RuntimeWarning: divide by zero encountered in divide
  D = (V_i - (log_div * V_f)) / (1 - log_div)  # Calculate D
C:\Users\Costanza\Documents\PhD\titan-processing-costanza\titan\prepr


-------
DEBUG:FOLDER N 0; EXP N 1; EXP_ID D20240404_E00_C00_F4500KHz_U_Lambda_T5

-------
DEBUG:FOLDER N 0; EXP N 2; EXP_ID D20240404_E00_C00_F4500KHz_U_Lambda_T6

-------
DEBUG:FOLDER N 0; EXP N 3; EXP_ID D20240405_E00_C00_F4500KHz_U_Lambda_T9

-------
DEBUG:FOLDER N 0; EXP N 4; EXP_ID D20240409_E00_C00_F4500KHz_U_Lambda_Neg_2

-------
DEBUG:FOLDER N 0; EXP N 5; EXP_ID D20240430_E00_C00_F4500KHz_U_Lambda_T14

-------
DEBUG:FOLDER N 0; EXP N 6; EXP_ID D20240430_E00_C00_F4500KHz_U_Lambda_T15

-------
DEBUG:FOLDER N 0; EXP N 7; EXP_ID D20240612_E00_C00_F4500KHz_U_Pilot_10w_01


C:\Users\Costanza\Documents\PhD\titan-processing-costanza\titan\preprocessing_functions.py:115: RuntimeWarning: divide by zero encountered in divide
  log_div = np.log((tau_i - C) / A) / np.log((tau_f - C) / A)  # Intermediate operation
C:\Users\Costanza\Documents\PhD\titan-processing-costanza\titan\preprocessing_functions.py:116: RuntimeWarning: invalid value encountered in divide
  D = (V_i - (log_div * V_f)) / (1 - log_div)  # Calculate D



-------
DEBUG:FOLDER N 0; EXP N 8; EXP_ID D20240620_E00_C00_F4500KHz_U_Pilot_10w_10

-------
DEBUG:FOLDER N 0; EXP N 9; EXP_ID D20240620_E00_C00_F4500KHz_U_Pilot_10w_11

-------
DEBUG:FOLDER N 0; EXP N 10; EXP_ID D20240627_E00_C00_F4500KHz_U_Pilot_10w_13

-------
DEBUG:FOLDER N 0; EXP N 11; EXP_ID D20240702_E00_C00_F4500KHz_U_Pilot_10w_40

-------
DEBUG:FOLDER N 0; EXP N 12; EXP_ID D20240710_E00_C00_F4500KHz_U_Pilot_10w_un

-------
DEBUG:FOLDER N 0; EXP N 13; EXP_ID D20240711_E00_C00_F4500KHz_U_Pilot_10w_26


In [ ]:
wb.save(Path(onedrive_path, "Master Data Folder", "Run Tracker.xlsx"))